# P23 — Prédire le prix d'une course de taxi vert à New York (TLC)

**Contexte métier.** La TLC de New York publie les courses des taxis « verts » (hors Manhattan central). Une application de réservation veut afficher une estimation de tarif honnête avant la course et détecter les courses anormales. La question métier : à partir de la distance, de la durée et de l'heure, quel montant au compteur attendre ?

**Objectif de l'analyse exploratoire.** Identifier la cible (`fare_amount`, la course hors taxes et suppléments) et les régresseurs, et repérer les pièges de facturation (tarifs plats JFK, annulations).

**Données.** NYC Open Data « 2019 Green Taxi Trip Data » (Socrata, TLC), export de 250 000 courses.

## 1. Téléchargement

Export Socrata du jeu « 2019 Green Taxi Trip Data » (TLC).

In [ ]:
import os, urllib.request

os.makedirs("data", exist_ok=True)
URL = "https://data.cityofnewyork.us/resource/q5mz-t52e.csv?$limit=250000"
if not os.path.exists("data/green_taxi_2019.csv"):
    urllib.request.urlretrieve(URL, "data/green_taxi_2019.csv")
print("OK", os.path.getsize("data/green_taxi_2019.csv") // (1024 * 1024), "Mo")

## 2. Chargement

Typage des dates, durée de course, nettoyage : le jeu contient des dates aberrantes (des courses 2009-2010 dans un jeu 2019) et des montants négatifs (annulations).

In [ ]:
import pandas as pd

df = pd.read_csv("data/green_taxi_2019.csv", low_memory=False)
print(df.shape, "lignes x colonnes")
manq = df.isna().mean().mul(100).round(1).sort_values(ascending=False)
print("% manquant :")
print(manq[manq > 0].to_string())

df["pickup_dt"] = pd.to_datetime(df["lpep_pickup_datetime"], errors="coerce")
df["dropoff_dt"] = pd.to_datetime(df["lpep_dropoff_datetime"], errors="coerce")
df["duree_min"] = (df["dropoff_dt"] - df["pickup_dt"]).dt.total_seconds() / 60
print()
print("Années de pickup (top) :")
print(df["pickup_dt"].dt.year.value_counts().head(5).to_string())
df = df[df["pickup_dt"].dt.year == 2019].copy()
df["heure"] = df["pickup_dt"].dt.hour
print()
print("Après filtrage 2019 :", len(df), "courses")
print(df[["fare_amount", "trip_distance"]].describe().round(2).to_string())

## 3. Analyse

Le tarif dépend d'abord du code tarifaire : standard (ratecode 1), forfait JFK (2, 52 $), Newark (3), Nassau/Westchester (4), groupe (5). On analyse les courses standard et la corrélation distance-tarif.

In [ ]:
print("Codes tarifaires :")
print(df["ratecodeid"].value_counts().to_string())
print()
print("Tarif médian par code :")
print(df.groupby("ratecodeid")["fare_amount"].median().to_string())
print()
std = df[(df["ratecodeid"] == 1) & (df["fare_amount"] > 0) & (df["trip_distance"] > 0)]
print("Courses standard (ratecode 1, tarif>0, distance>0) :", len(std))
print("Corrélations Spearman avec fare_amount :")
print(std[["fare_amount", "trip_distance", "duree_min", "passenger_count"]].corr(method="spearman")["fare_amount"].round(2).to_string())
print()
d = std[(std["trip_distance"] >= 0.5) & (std["trip_distance"] <= 30)]
print("$ par mile (médiane) :", round((d["fare_amount"] / d["trip_distance"]).median(), 2))
print("Tarifs négatifs (toutes courses) :", round(100 * (df["fare_amount"] < 0).mean(), 1), "% ;",
      "distances nulles :", round(100 * (df["trip_distance"] == 0).mean(), 1), "%")

## 4. Résultats observés (exécution du 16/10/2026)

- 250 000 courses, 20 colonnes ; le jeu contient des dates aberrantes (de 2009) : après filtrage 2019 il reste ~249 500 courses
- codes tarifaires : 1 standard 226 802 ; 5 groupe 22 114 ; 2 JFK 343 ; 4 Nassau/Westchester 132 ; 3 Newark 70
- tarif médian par code : standard 9,50 $, JFK 52 $ (forfait), Newark 22,50 $, groupe 24,10 $
- fare_amount : médiane 10 $, 5 % : 4 $, 95 % : 39,99 $, max 330 $ ; 0,4 % de négatifs, 0,2 % de nuls
- trip_distance : médiane 2,0 mile, 95 % : 12,4 ; 1,6 % de distances nulles
- corrélations Spearman avec le tarif (courses standard) : distance 0,96, durée 0,96
- $ par mile (médiane, courses standard 0,5-30 miles) : 5,0
- manquants : ehail_fee 100 % ; 0,2 % de NaN sur vendor/ratecode/payment/trip_type (1 231 courses)
- total_amount = fare + taxes + péages + surcharges : le pourboire n'est compté que pour les paiements carte

## 5. Variable cible et régresseurs

**Cible (régression) : `fare_amount`** de la course, en dollars, sur les courses standard (ratecode 1, tarif > 0, distance > 0) — un problème quasi linéaire, idéal pour un premier modèle classique (régression linéaire puis gradient boosting).

**Régresseurs retenus :**
- `trip_distance` (corr. 0,96) et `duree_min` (0,96) : les deux composantes du compteur
- `heure`, jour de semaine (suppléments heure de pointe / nuit)
- `pulocationid` / `dolocationid` (zones TLC) : effet géographique
- `ratecodeid` : à modéliser séparément (forfaits JFK/Newark) ou à exclure du périmètre
- `passenger_count` (sans effet net, corr. -0,02)

**Pièges :**
- forfaits : ratecode 2 (JFK, 52 $) et 3 (Newark) cassent la relation distance-tarif : segmenter
- montants négatifs (0,4 %) : annulations/remboursements, à écarter ; distances nulles 1,6 % : GPS raté
- dates aberrantes (courses 2009 dans le jeu 2019) : filtrer sur l'année
- `total_amount` n'est pas la cible : il ajoute taxes et péages ; et le pourboire n'est compté que pour paiement par carte
- baseline à battre : 2,50 $ + 5 $/mile x distance — un modèle n'a d'intérêt que s'il capture l'heure et la durée